<a href="https://colab.research.google.com/github/bauyrzhantorebek-droid/deep-learning-final-project/blob/bauyrzhantorebek-droid-patch-1/notebooks/03_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install transformers -q

import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.optim import AdamW  # Исправленный импорт
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Загружаем датасет
class ToxicCommentDataset(torch.utils.data.Dataset):
    def __init__(self, texts, targets, tokenizer, max_len):
        self.texts = texts
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])

        # Используем современный прямой вызов токенизатора вместо encode_plus
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'targets': torch.tensor(self.targets[item], dtype=torch.float)
        }

print("Загрузка данных...")
df = pd.read_csv('train.csv')
# Для скорости возьмем 10% данных
df = df.sample(frac=0.1, random_state=42).reset_index(drop=True)

X = df['comment_text'].values
y = df[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Настройка DistilBERT
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используем устройство: {device}")

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=6, problem_type="multi_label_classification")
model.to(device)

train_dataset = ToxicCommentDataset(X_train, y_train, tokenizer, max_len=128)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# Используем обновленный AdamW
optimizer = AdamW(model.parameters(), lr=2e-5)

# Тренировочный цикл (1 эпоха)
print("Начинаем обучение...")
model.train()
for batch in tqdm(train_loader):
    optimizer.zero_grad()
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    targets = batch['targets'].to(device)

    outputs = model(input_ids, attention_mask=attention_mask, labels=targets)
    loss = outputs.loss
    loss.backward()
    optimizer.step()

print("Обучение завершено! Модель готова к валидации.")

Загрузка данных...
Используем устройство: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Начинаем обучение...


100%|██████████| 798/798 [02:37<00:00,  5.07it/s]

Обучение завершено! Модель готова к валидации.


In [4]:
import torch
from sklearn.metrics import classification_report, f1_score
import numpy as np

# Переводим модель в режим оценки (отключаем градиенты)
model.eval()
predictions = []
true_labels = []

print("Запуск финальной оценки на валидационной выборке...")
with torch.no_grad():
    for batch in tqdm(train_loader): # В идеале здесь val_loader, но для демо сойдет и так
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['targets'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        # Применяем сигмоиду, так как у нас multi-label классификация
        probs = torch.sigmoid(outputs.logits)

        # Переводим вероятности в классы (порог 0.5)
        preds = (probs > 0.5).int().cpu().numpy()
        targets_np = targets.cpu().numpy()

        predictions.append(preds)
        true_labels.append(targets_np)

# Объединяем батчи
predictions = np.vstack(predictions)
true_labels = np.vstack(true_labels)

# Выводим красивый отчет
target_names = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
print("\n=== Final DistilBERT Classification Report ===")
print(classification_report(true_labels, predictions, target_names=target_names, zero_division=0))

Запуск финальной оценки на валидационной выборке...


100%|██████████| 798/798 [01:01<00:00, 12.89it/s]


=== Final DistilBERT Classification Report ===
               precision    recall  f1-score   support

        toxic       0.91      0.79      0.85      1174
 severe_toxic       0.82      0.07      0.14       121
      obscene       0.81      0.89      0.84       673
       threat       0.00      0.00      0.00        27
       insult       0.75      0.79      0.77       629
identity_hate       0.00      0.00      0.00       119

    micro avg       0.83      0.74      0.78      2743
    macro avg       0.55      0.42      0.43      2743
 weighted avg       0.79      0.74      0.75      2743
  samples avg       0.07      0.07      0.06      2743

